# Image feature extraction: clip

CLIP ViT-B/16 vision tower, contrastive image-text pretraining.

Runs every one of the 1,513 verified, transform-ready images (`resolved_image_keys.json`, produced
by `preprocessing/images_preprocessing.ipynb`) through this **one** frozen encoder and saves the
resulting feature vectors. This notebook only produces data for `clip` -- the other three
candidates (part of the deliberate four-paradigm backbone screen for the image side,
`src/models/image_encoders.py`) each have their own identical-in-structure notebook alongside this
one in `data_generation/`, so any one of the four can be re-run independently without touching the
others.

No missing/corrupted-image handling here -- that already happened in
`images_preprocessing.ipynb`, and `resolved_image_keys.json` is guaranteed to contain only images
that already opened and transformed successfully. If this notebook still hits a failure, that is a
new problem (e.g. a file that went missing since that check), not an expected one, and is reported
loudly rather than silently skipped.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../../..')  # local: this notebook's own project root
                                          # (data_generation -> 3_multimodal -> experiments -> 复现 -> Dissertation)

REPO_ROOT = PROJECT_ROOT / '复现'
DATA_ROOT = PROJECT_ROOT / 'CGMacros'  # raw participant folders + photos/ live here
PROCESSED_DIR = REPO_ROOT / 'preprocessing' / 'processed_native5min'
sys.path.insert(0, str(REPO_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={REPO_ROOT}  DATA_ROOT={DATA_ROOT}  "
      f"PROCESSED_DIR={PROCESSED_DIR}")

In [ ]:
import json
import pickle
import time

import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm

from src.models.image_encoders import load_encoder, IMG_TRANSFORM

ENCODER_NAME = 'gated_clip'
BATCH_SIZE = 64

DEVICE = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available()
          else 'cpu')
print(f'device: {DEVICE}')

FEATURES_DIR = PROCESSED_DIR / 'img_features'
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = FEATURES_DIR / f'img_features_{ENCODER_NAME}.pkl'

## Load the verified image list

Reads `resolved_image_keys.json` -- every key here is guaranteed openable and transform-ready
(`images_preprocessing.ipynb`'s own coverage check confirmed `known_failure=0`). Paths are rebuilt
from `DATA_ROOT` + key, not stored directly, so this works unchanged on Colab.

In [ ]:
resolved_keys_path = PROCESSED_DIR / 'img_preprocessing' / 'resolved_image_keys.json'
with open(resolved_keys_path) as f:
    resolved_keys = json.load(f)

image_paths = {}
for key in resolved_keys:
    participant_id, filename = key.split('/', 1)
    image_paths[key] = DATA_ROOT / participant_id / 'photos' / filename

print(f'{len(image_paths)} verified images to encode')

## Extract clip features

One pass, batched, with a visible progress bar and a concrete sample output (not just a final
count) so it's clear the encoder is actually producing something meaningful, not just running.

In [ ]:
model, feat_dim, forward_fn = load_encoder(ENCODER_NAME, device=DEVICE)
print(f'{ENCODER_NAME}  feat_dim={feat_dim}')

keys = list(image_paths.keys())
features = {}
skipped = []

t0 = time.time()
pbar = tqdm(range(0, len(keys), BATCH_SIZE), desc=ENCODER_NAME)
for i in pbar:
    batch_keys = keys[i: i + BATCH_SIZE]
    imgs, valid_keys = [], []
    for k in batch_keys:
        try:
            img = Image.open(image_paths[k]).convert('RGB')
            imgs.append(IMG_TRANSFORM(img))
            valid_keys.append(k)
        except Exception as e:
            skipped.append((k, str(e)))

    if not imgs:
        continue

    batch_tensor = torch.stack(imgs).to(DEVICE)
    with torch.no_grad():
        feats = forward_fn(model, batch_tensor).cpu().numpy()

    for k, feat in zip(valid_keys, feats):
        features[k] = feat.astype(np.float32)

    pbar.set_postfix(encoded=len(features), skipped=len(skipped))

elapsed = time.time() - t0
del model
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

print(f'\nencoded {len(features)}/{len(keys)} images in {elapsed:.0f}s')
if skipped:
    print(f'\u26a0\ufe0f UNEXPECTED: {len(skipped)} keys failed despite being pre-verified -- '
          f'investigate, do not ignore:')
    print('  ', skipped[:5], '...' if len(skipped) > 5 else '')
else:
    print('0 skipped, as expected (all keys were pre-verified by images_preprocessing.ipynb)')

# Sample output: one concrete vector, not just a count.
_demo_key = keys[0]
_demo_vec = features[_demo_key]
print(f"\nsample output -> key='{_demo_key}'")
print(f'  shape={_demo_vec.shape}  dtype={_demo_vec.dtype}  '
      f'first5={np.round(_demo_vec[:5], 3)}  L2norm={np.linalg.norm(_demo_vec):.2f}')

## Save

In [ ]:
with open(OUT_PATH, 'wb') as f:
    pickle.dump(features, f)

print(f'saved {len(features)} x {feat_dim}-d {ENCODER_NAME} features to {OUT_PATH} '
      f'({OUT_PATH.stat().st_size / 1e6:.1f} MB)')

assert len(features) == len(resolved_keys), (
    f'expected {len(resolved_keys)} features, got {len(features)} -- investigate before using '
    f'this file downstream'
)
print('count matches resolved_image_keys.json exactly -- OK')